# Mechanistic Anomaly DetectionBuild baseline activation profiles and detect when internal computation deviates from typical patterns. Useful for safety auditing and identifying adversarial inputs.The key insight: checking *how* the model computes (its internal pathway) can reveal anomalies invisible in the output alone.

## Why This Matters

Mechanistic anomaly detection identifies unusual activation patterns that may indicate trojans, backdoors, or unexpected behaviors. By establishing baseline activation profiles and flagging deviations, this connects interpretability directly to model safety and security.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformercfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

In [ ]:
from irtk.mechanistic_anomaly_detection import (    build_activation_profile,    detect_pathway_anomalies,    find_trojan_signatures,    compare_surface_vs_internals,    cluster_computational_strategies,)

## Building Activation ProfilesCreate a baseline of "normal" computation from reference inputs.

In [ ]:
reference_inputs = [jnp.array([i, i+1, i+2, i+3]) for i in range(10)]profile = build_activation_profile(model, reference_inputs)print(f"Profiled {profile['n_samples']} inputs across {len(profile['hook_names'])} hooks")for name in profile['hook_names']:    print(f"  {name}: mean_norm={np.linalg.norm(profile['means'][name]):.4f}, "          f"mean_std={np.mean(profile['stds'][name]):.4f}")

## Detecting Pathway AnomaliesScore how anomalous a new input's computation is compared to the baseline.

In [ ]:
novel_tokens = jnp.array([40, 41, 42, 43])result = detect_pathway_anomalies(model, novel_tokens, profile)print(f"Total anomaly score: {result['total_anomaly_score']:.4f}")print(f"Most anomalous layer: {result['max_anomaly_layer']}")for name, score in result['layer_anomaly_scores'].items():    print(f"  {name}: anomaly = {score:.4f}")

## Trojan Signature DetectionLook for sparse, localized anomalous activations — a pattern consistent with backdoor circuits that activate on specific triggers.

In [ ]:
result = find_trojan_signatures(model, novel_tokens, profile)print(f"Max z-score: {result['max_z_score']:.4f}")print(f"Trojan risk score: {result['trojan_risk_score']:.4f}")for name, dims in result['suspicious_dimensions'].items():    if dims:        print(f"  {name}: {len(dims)} suspicious dims")

## Surface vs Internal PredictionsDoes the model's internal trajectory agree with its final output? Disagreement could indicate deceptive computation.

In [ ]:
result = compare_surface_vs_internals(model, tokens)print(f"Final prediction: token {result['final_prediction']}")print(f"Agreement fraction: {result['agreement_fraction']:.2f}")print(f"Disagreement layers: {result['disagreement_layers']}")print(f"Trajectory consistency: {result['trajectory_consistency']:.4f}")for l, pred in enumerate(result['layer_predictions']):    match = "match" if pred == result['final_prediction'] else "DIFFER"    print(f"  Layer {l}: predicts token {pred} [{match}]")

## Clustering Computational StrategiesGroup inputs by how the model processes them internally, revealing qualitatively different computation paths.

In [ ]:
inputs = [jnp.array([i, i+1, i+2, i+3]) for i in range(12)]result = cluster_computational_strategies(model, inputs, n_clusters=3)print(f"Cluster sizes: {result['cluster_sizes']}")print(f"Between-cluster variance: {result['between_cluster_variance']:.6f}")for k in range(len(result['cluster_sizes'])):    print(f"  Cluster {k}: {result['cluster_sizes'][k]} inputs, "          f"within-var={result['within_cluster_variance'][k]:.6f}")